In [0]:
## Version 1.0
# Customer Churn Prediction with Databricks

End-to-end machine learning pipeline built using Databricks, Delta Lake, PySpark ML and MLflow.

## Objectives

- Build a Medallion Architecture pipeline
- Prepare data for machine learning
- Train churn prediction models
- Track experiments with MLflow
- Compare multiple algorithms


## Bronze Layer

The Bronze layer contains the raw ingested data without transformations.

Purpose:

- Preserve source data
- Enable reproducibility
- Support data lineage

In [0]:
df = spark.table(
    "workspace.default.wa_fn_use_c_telco_customer_churn"
)

display(df)

In [0]:
print(df.count())

In [0]:
print(len(df.columns))

In [0]:
df.printSchema()

In [0]:
display(
    df.groupBy("Churn").count()
)

In [0]:
df = spark.table(
    "workspace.default.wa_fn_use_c_telco_customer_churn"
)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.bronze_customer_churn"
    )

In [0]:
display(
    spark.table(
        "workspace.default.bronze_customer_churn"
    )
)


## Silver Layer

Data cleaning operations:

- Removed duplicates
- Cast TotalCharges to numeric
- Removed invalid records

In [0]:
bronze = spark.table(
    "workspace.default.bronze_customer_churn"
)

silver = (
    bronze
    .dropDuplicates()
)

silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.silver_customer_churn"
    )

In [0]:
silver = spark.table(
    "workspace.default.silver_customer_churn"
)

print(f"Rows: {silver.count()}")
print(f"Columns: {len(silver.columns)}")

In [0]:
display(
    silver.groupBy("Churn")
          .count()
)

In [0]:
silver.printSchema()

In [0]:
from pyspark.sql.functions import col, trim

silver = spark.table(
    "workspace.default.silver_customer_churn"
)

silver = silver.withColumn(
    "TotalCharges",
    trim(col("TotalCharges"))
)

silver = silver.filter(
    col("TotalCharges") != ""
)

silver = silver.withColumn(
    "TotalCharges",
    col("TotalCharges").cast("double")
)

silver.printSchema()

In [0]:
categorical_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

In [0]:
from pyspark.ml.feature import StringIndexer

indexers = [
    StringIndexer(
        inputCol=col_name,
        outputCol=f"{col_name}_idx",
        handleInvalid="keep"
    )
    for col_name in categorical_cols
]

In [0]:
churn_indexer = StringIndexer(
    inputCol="Churn",
    outputCol="label"
)

In [0]:
from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder(
    inputCols=[
        f"{c}_idx"
        for c in categorical_cols
    ],
    outputCols=[
        f"{c}_ohe"
        for c in categorical_cols
    ]
)

In [0]:
numeric_cols = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=
        [f"{c}_ohe" for c in categorical_cols]
        + numeric_cols,
    outputCol="features"
)

In [0]:
from pyspark.ml import Pipeline

pipeline = Pipeline(
    stages=
        indexers
        + [churn_indexer]
        + [encoder]
        + [assembler]
)

In [0]:
gold = pipeline.fit(silver).transform(silver)

gold.select(
    "features",
    "label"
).show(5, truncate=False)

In [0]:
silver.count()

In [0]:
silver = spark.table(
    "workspace.default.silver_customer_churn"
)

In [0]:
from pyspark.sql.functions import col, trim

silver = silver.withColumn(
    "TotalCharges",
    trim(col("TotalCharges"))
)

silver = silver.filter(
    col("TotalCharges") != ""
)

silver = silver.withColumn(
    "TotalCharges",
    col("TotalCharges").cast("double")
)

In [0]:
from pyspark.ml.feature import StringIndexer

categorical_cols = [
    "gender",
    "Partner",
    "InternetService",
    "Contract",
    "PaymentMethod"
]

In [0]:
indexers = []

for c in categorical_cols:
    indexers.append(
        StringIndexer(
            inputCol=c,
            outputCol=f"{c}_idx",
            handleInvalid="keep"
        )
    )

In [0]:
churn_indexer = StringIndexer(
    inputCol="Churn",
    outputCol="label"
)

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "gender_idx",
        "Partner_idx",
        "InternetService_idx",
        "Contract_idx",
        "PaymentMethod_idx",
        "tenure",
        "MonthlyCharges",
        "TotalCharges"
    ],
    outputCol="features"
)

In [0]:
from pyspark.ml import Pipeline

pipeline = Pipeline(
    stages=indexers + [churn_indexer, assembler]
)

In [0]:
pipeline_model = pipeline.fit(silver)

gold = pipeline_model.transform(silver)

gold.select(
    "features",
    "label"
).show(5, truncate=False)

+----------------------------------------+-----+
|features                                |label|
+----------------------------------------+-----+
|[0.0,1.0,1.0,1.0,3.0,72.0,90.25,6369.45]|0.0  |
|[0.0,0.0,1.0,1.0,3.0,52.0,79.75,4217.8] |0.0  |
|[1.0,1.0,2.0,0.0,0.0,1.0,19.45,19.45]   |0.0  |
|[1.0,0.0,2.0,0.0,1.0,6.0,20.7,112.75]   |0.0  |
|[0.0,0.0,2.0,1.0,1.0,57.0,19.6,1170.55] |0.0  |
+----------------------------------------+-----+
only showing top 5 rows


In [0]:
del pipeline_model

In [0]:
train, test = gold.randomSplit(
    [0.8, 0.2],
    seed=42
)

print(train.count())
print(test.count())

5690
1342


In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)

model = lr.fit(train)

In [0]:
predictions = model.transform(test)

display(
    predictions.select(
        "label",
        "prediction",
        "probability"
    )
)

label,prediction,probability
1.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.2851747135201621"",""0.7148252864798379""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.865245811632011"",""0.13475418836798903""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.561886824933542"",""0.43811317506645797""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8852516247115246"",""0.11474837528847537""]}"
1.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.47877631466086523"",""0.5212236853391348""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7874987533334656"",""0.21250124666653436""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9959046685833443"",""0.004095331416655745""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9743288633150683"",""0.025671136684931706""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9690859505141121"",""0.03091404948588794""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8723653356367976"",""0.12763466436320237""]}"


In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

acc = accuracy.evaluate(predictions)

print(f"Accuracy: {acc:.4f}")

Accuracy: 0.8018


In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

auc_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = auc_eval.evaluate(predictions)

print(f"AUC: {auc:.4f}")

AUC: 0.8405


In [0]:
import mlflow

mlflow.__version__

'3.8.1'

In [0]:
import mlflow

with mlflow.start_run():

    mlflow.log_param(
        "model",
        "logistic_regression"
    )

    mlflow.log_metric(
        "accuracy",
        acc
    )

    mlflow.log_metric(
        "auc",
        auc
    )

In [0]:
import mlflow

with mlflow.start_run() as run:

    mlflow.log_param("model", "logistic_regression")
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("auc", auc)

    print("Run ID:", run.info.run_id)
    

Run ID: 6734c4c9b393481882e4bac1d7d6eead


In [0]:
mlflow.search_runs()

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.accuracy,metrics.auc,params.model,tags.mlflow.databricks.cluster.id,tags.mlflow.databricks.notebookRevisionID,tags.mlflow.databricks.notebookID,tags.mlflow.databricks.cluster.info,tags.mlflow.source.name,tags.mlflow.user,tags.mlflow.databricks.workspaceID,tags.mlflow.databricks.notebookPath,tags.mlflow.runName,tags.mlflow.runColor,tags.mlflow.databricks.notebook.commandID,tags.mlflow.databricks.webappURL,tags.mlflow.databricks.workspaceURL,tags.mlflow.source.type,tags.mlflow.databricks.cluster.libraries
0,6734c4c9b393481882e4bac1d7d6eead,2603474668618193,FINISHED,dbfs:/databricks/mlflow-tracking/2603474668618...,2026-06-18 15:19:59.115000+00:00,2026-06-18 15:19:59.443000+00:00,0.801788,0.840498,logistic_regression,0618-150042-dp4li9hj-v2n,1781795999546,2603474668618193,"{""cluster_name"":"""",""spark_version"":""client.5.6...",/Users/js.enciso103@gmail.com/01_data_ingestion,js.enciso103@gmail.com,7474644342814056,/Users/js.enciso103@gmail.com/01_data_ingestion,intelligent-colt-336,#da4c4c,1001781794724509_8855973280813574544_ef39ed003...,https://dbc-46cc8512-b600.cloud.databricks.com,https://dbc-46cc8512-b600.cloud.databricks.com,NOTEBOOK,"{""installable"":[],""redacted"":[]}"
1,4d58efd1ed4e499ba717c76f1721912e,2603474668618193,FINISHED,dbfs:/databricks/mlflow-tracking/2603474668618...,2026-06-18 15:18:52.344000+00:00,2026-06-18 15:18:52.764000+00:00,0.801788,0.840498,logistic_regression,0618-150042-dp4li9hj-v2n,1781795932846,2603474668618193,"{""cluster_name"":"""",""spark_version"":""client.5.6...",/Users/js.enciso103@gmail.com/01_data_ingestion,js.enciso103@gmail.com,7474644342814056,/Users/js.enciso103@gmail.com/01_data_ingestion,dazzling-bug-389,#5387dd,1001781794724509_8668920313427788348_544e94473...,https://dbc-46cc8512-b600.cloud.databricks.com,https://dbc-46cc8512-b600.cloud.databricks.com,NOTEBOOK,"{""installable"":[],""redacted"":[]}"


In [0]:
mlflow.get_tracking_uri()

'databricks'

In [0]:
import mlflow

experiment = mlflow.get_experiment_by_name("/Users/me")
print(experiment)

None


In [0]:
print(mlflow.get_experiment())

---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
File <command-4896767763279034>, line 1
----> 1 print(mlflow.get_experiment())

TypeError: get_experiment() missing 1 required positional argument: 'experiment_id'

In [0]:
import mlflow

print(mlflow.search_experiments())

[<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/2603474668618193', creation_time=1781795932415, experiment_id='2603474668618193', last_update_time=1781795999115, lifecycle_stage='active', name='/Users/js.enciso103@gmail.com/01_data_ingestion', tags={'mlflow.experiment.sourceName': '/Users/js.enciso103@gmail.com/01_data_ingestion',
 'mlflow.experimentType': 'NOTEBOOK',
 'mlflow.ownerEmail': 'js.enciso103@gmail.com',
 'mlflow.ownerId': '74046866595976'}>]


In [0]:
runs = mlflow.search_runs()

print(runs)

                             run_id  ... tags.mlflow.databricks.cluster.libraries
0  6734c4c9b393481882e4bac1d7d6eead  ...         {"installable":[],"redacted":[]}
1  4d58efd1ed4e499ba717c76f1721912e  ...         {"installable":[],"redacted":[]}

[2 rows x 24 columns]


In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

for exp in client.search_experiments():
    print(
        f"ID: {exp.experiment_id}",
        f"Name: {exp.name}"
    )

ID: 2603474668618193 Name: /Users/js.enciso103@gmail.com/01_data_ingestion


In [0]:
print(f"Accuracy: {acc:.4f}")
print(f"AUC: {auc:.4f}")

Accuracy: 0.8018
AUC: 0.8405


In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    seed=42
)

rf_model = rf.fit(train)

rf_predictions = rf_model.transform(test)

rf_auc = auc_eval.evaluate(rf_predictions)

print(f"RF AUC: {rf_auc:.4f}")

RF AUC: 0.8457
